# GreenNode - Entrenamiento del clasificador de residuos

Notebook para Google Colab. Entrena un modelo **MobileNetV2 + transfer learning** sobre el dataset **garbage-classification (12 clases, ~15.150 imagenes)** para clasificar residuos en 6 categorias, siguiendo `docs/AI_MODEL.md`.

**Salidas:**
- `waste_classifier_v1.tflite` -> app movil y web (se copia a `web/public/model/`)
- `labels.json` -> orden de las clases

**Antes de empezar:** `Entorno de ejecucion -> Cambiar tipo de entorno -> GPU (T4)`.

Ejecuta las celdas de arriba a abajo.

## 1. Instalar dependencias

In [ ]:
# TensorFlow ya viene preinstalado en Colab. Solo instalamos lo que falta.
# (No usamos tensorflowjs: exportamos a .tflite y lo usamos directo en la web.)
!pip install -q kagglehub scikit-learn

## 2. Imports y configuracion

El orden de clases (`CLASS_ORDER`) debe coincidir EXACTAMENTE con el enum `WasteType` de la app (`src/domain/entities/WasteClassification.ts`).

In [ ]:
import json, os, pathlib, shutil
import numpy as np
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 6
SEED = 123
EPOCHS_PHASE1 = 10   # feature extraction
EPOCHS_PHASE2 = 10   # fine-tuning
FINE_TUNE_AT = -50   # descongelar ultimas 50 capas

CLASS_ORDER = ['organic', 'plastic', 'paper', 'glass', 'metal', 'special']
print('Clases:', CLASS_ORDER)

## 3. Descargar el dataset (12 clases, ~15.150 imagenes)

Usamos `mostafaabla/garbage-classification`: 12 clases que incluyen
'biological' (organico) y muchas mas imagenes que TrashNet.
Se descarga con `kagglehub`. Puede pedirte autenticarte con Kaggle la primera vez.

In [ ]:
import kagglehub
dataset_path = kagglehub.dataset_download('mostafaabla/garbage-classification')

# La carpeta con subcarpetas por clase suele ser 'garbage_classification'.
# Buscamos automaticamente la carpeta que contenga las 12 clases.
base = pathlib.Path(dataset_path)
candidates = [base] + [p for p in base.rglob('*') if p.is_dir()]
RAW_DIR = None
for c in candidates:
    subs = [d.name for d in c.iterdir() if d.is_dir()] if c.is_dir() else []
    if 'biological' in subs and 'plastic' in subs:
        RAW_DIR = c
        break
if RAW_DIR is None:
    RAW_DIR = base
print('Dataset en:', RAW_DIR)
print('Subcarpetas:', sorted([p.name for p in RAW_DIR.iterdir() if p.is_dir()]))

## 4. Reorganizar las 12 clases a las 6 del proyecto

Mapeamos las 12 categorias del dataset a las 6 de GreenNode.
`biological` -> `organic` (por fin tenemos organico). Los 3 tipos de vidrio
se unen en `glass`. Baterias, ropa, zapatos y trash van a `special`.

In [ ]:
TRASHNET_TO_PROJECT = {
    'biological': 'organic',
    'plastic': 'plastic',
    'paper': 'paper',
    'cardboard': 'paper',
    'green-glass': 'glass',
    'brown-glass': 'glass',
    'white-glass': 'glass',
    'metal': 'metal',
    'battery': 'special',
    'clothes': 'special',
    'shoes': 'special',
    'trash': 'special',
}

DATA_DIR = pathlib.Path('greennode_dataset')
DATA_DIR.mkdir(exist_ok=True)
for cls in CLASS_ORDER:
    (DATA_DIR / cls).mkdir(exist_ok=True)

# Nombres alternativos por si el dataset usa singular/plural distinto
ALIASES = {'battery': ['battery', 'batteries'], 'clothes': ['clothes', 'clothing'],
           'trash': ['trash', 'garbage']}
EXTS = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')

copied = 0
for src_class, dst_class in TRASHNET_TO_PROJECT.items():
    posibles = ALIASES.get(src_class, [src_class])
    src = next((RAW_DIR / p for p in posibles if (RAW_DIR / p).exists()), None)
    if src is None:
        print(f'  [aviso] no existe carpeta para {src_class}')
        continue
    for ext in EXTS:
        for img in src.glob(ext):
            shutil.copy(img, DATA_DIR / dst_class / f'{src_class}_{img.name}')
            copied += 1
print(f'Imagenes copiadas: {copied}')
for cls in CLASS_ORDER:
    n = sum(len(list((DATA_DIR / cls).glob(e))) for e in EXTS)
    print(f'  {cls}: {n}')

## 5. Cargar datos (train / validation)

In [ ]:
# Excluye clases vacias (cuenta todas las extensiones de imagen)
def _count(d):
    return sum(len(list(d.glob(e))) for e in ('*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG'))
present = [c for c in CLASS_ORDER if _count(DATA_DIR / c) > 0]
print('Clases con datos:', present)

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='training', seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, class_names=present)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='validation', seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, class_names=present)

NUM_CLASSES = len(present)
with open('labels.json', 'w') as f:
    json.dump(present, f)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

## 6. Data augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name='data_augmentation')

## 7. Construir el modelo (MobileNetV2 + transfer learning)

El preprocesamiento de MobileNetV2 escala los pixeles al rango [-1, 1]. La app aplica el MISMO preprocesamiento antes de inferir.

In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 8. Fase 1 - Feature extraction (base congelada)

In [ ]:
history1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_PHASE1)

## 9. Fase 2 - Fine-tuning (ultimas 50 capas)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history2 = model.fit(train_ds, validation_data=val_ds,
                     epochs=EPOCHS_PHASE1 + EPOCHS_PHASE2,
                     initial_epoch=history1.epoch[-1] + 1)

## 10. Evaluacion (metricas + matriz de confusion)

Estos resultados los puedes usar directamente en la monografia.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=present, zero_division=0))
print('Matriz de confusion:')
print(confusion_matrix(y_true, y_pred))

## 11. Exportar a TFLite (app movil)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()
with open('waste_classifier_v1.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'TFLite guardado ({len(tflite_model)/1024/1024:.2f} MB)')

## 12. Exportar a TensorFlow.js (web)

In [ ]:
# NOTA: Ya NO exportamos a TensorFlow.js.
# La libreria tensorflowjs esta rota en Python 3.13 / numpy nuevo de Colab.
# En su lugar, la WEB usa directamente el archivo .tflite (paso 11) con la
# libreria @tensorflow/tfjs-tflite. No necesitas hacer nada en este paso.
print('Paso omitido: la web usa el .tflite directamente. Continua al paso 13.')

## 13. Descargar los archivos

Descargalos ANTES de cerrar la sesion (el almacenamiento de Colab es temporal).

Luego: descomprime `tfjs_model.zip` y copia su contenido a `web/public/model/`.

In [ ]:
from google.colab import files
files.download('waste_classifier_v1.tflite')
files.download('labels.json')